# 07p — Final methodological synthesis

This stage closes the CP2-v2 HOME/OFFICE validation track by synthesizing already-measured aggregate evidence.

It does **not** rerun raw GPS processing, stay detection, clustering, semantic inference, historical POI extraction, or imagery review.

```text
Stage 05   internal reliability
Stage 07i  threshold sensitivity
Stage 07j  OFFICE near-miss robustness
Stage 07l  imagery + behavior + BCL synthesis
Stage 07m  Trackintel semantic-only parity
Stage 07n  Trackintel end-to-end sensitivity
Stage 07o  literature comparator suite
        ↓
Stage 07p  final evidence hierarchy + policy boundary
```

Final production reference entering this stage:

```text
HOME = 27
OFFICE = 16
```

The notebook consumes only aggregate CSVs from the existing Modal Volume. No user-level private data are required.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))
CACHE_ROOT=Path('/mnt/geolife-data/cache/cp2_v2')
C=CACHE_ROOT/'07p_methodological_synthesis'
C.mkdir(parents=True,exist_ok=True)

def run(cmd,cwd=None):
    subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)

for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path:
        sys.path.insert(0,p)

spec=importlib.util.spec_from_file_location(
    's07p',
    REPO_DIR/'analysis'/'07p_methodological_synthesis.py',
)
assert spec is not None and spec.loader is not None
s07p=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07p
spec.loader.exec_module(s07p)

print('sha:',subprocess.run(
    ['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],
    capture_output=True,text=True,check=True
).stdout.strip())
print(s07p.synthetic_self_check())

## 1. Load measured aggregate lineage

Required aggregate roots:

```text
05_home_office_reliability_validation
07i_threshold_sensitivity
07j_near_miss_office_audit
07l_imagery_unblinding_synthesis
07m_trackintel_semantic_parity
07n_trackintel_end_to_end
07o_literature_comparator_suite
```

Stage 07p hard-stops if a required aggregate file is missing.

In [ ]:
frames=s07p.load_aggregate_inputs(CACHE_ROOT)

input_inventory=pd.DataFrame([
    {
        'key':name,
        'rows':len(frame),
        'columns':len(frame.columns),
    }
    for name,frame in sorted(frames.items())
])
display(input_inventory)

## 2. Validate measured lineage

Before synthesizing, reproduce the already-accepted measurements from the cached aggregate tables.

Examples include:

- Stage-05 HOME/OFFICE cross-method, held-out and dropout values;
- Stage-07i frozen/relaxed OFFICE counts;
- Stage-07j near-miss robustness counts;
- Stage-07l imagery/BCL policy snapshot;
- Stage-07m OSNA parity;
- Stage-07n stay/end-to-end decomposition;
- Stage-07o literature agreement.

This is a consistency gate, not a new experiment.

In [ ]:
lineage_validation=s07p.validate_measured_inputs(frames)
assert lineage_validation['passed'].all()
print('measured lineage validation: PASS')
display(lineage_validation)

## 3. HOME evidence ladder

HOME should be interpreted through multiple evidence families rather than one score.

The important question is whether candidate identity converges under:

- internal semantic rankers;
- held-out/dropout perturbation;
- independent open-source semantics;
- independent end-to-end representation;
- literature-style temporal/spatial heuristics.

In [ ]:
home_evidence=s07p.build_home_evidence_summary(frames)
display(home_evidence)

## 4. OFFICE evidence ladder

OFFICE uses the same evidence hierarchy, but the expected result is more method-sensitive.

Strong agreement from one comparator is not enough to widen production because:

- recurrence-based OFFICE identity is weaker;
- held-out persistence is weaker;
- end-to-end representation changes alter candidate identity more;
- broad external heuristics often select many more WORK users than production.

In [ ]:
office_evidence=s07p.build_office_evidence_summary(frames)
display(office_evidence)

## 5. OFFICE near-miss policy synthesis

Stage 07i established that the frozen OFFICE gate is conservative in **coverage**:

```text
baseline                16
margin .10 → .05        18   (+2)
share  .30 → .20        23   (+7)
```

Stages 07j and 07l then tested whether those nine one-step additions were robust enough to justify a production-wide relaxation.

This table contains the decisive persistence/context evidence, not a composite score.

In [ ]:
near_miss_policy=s07p.build_near_miss_policy_summary(frames)
display(near_miss_policy)

## 6. End-to-end implementation sensitivity

Stage 07m showed what happens when stays and locations are held fixed.

Stage 07n showed what happens when Trackintel owns the upstream pipeline.

The synthesis must keep those questions separate:

```text
semantic-rule disagreement
≠
stay-extraction disagreement
≠
clustering disagreement
```

In [ ]:
pipeline_sensitivity=s07p.build_pipeline_sensitivity_summary(frames)
display(pipeline_sensitivity)

## 7. Literature / feature-space context

Stage 07o intentionally mixes different comparator contracts:

- exact candidate identity for same-namespace temporal rankers;
- spatial correspondence for the geohash namespace;
- descriptive important-location features for Pavan.

The raw aggregate tables remain visible here so the final conclusion does not erase coverage differences.

In [ ]:
print('LITERATURE SUITE')
display(frames['07o_suite'])

print('PAVAN FEATURE MEDIANS')
display(frames['07o_pavan_feature'])

print('PAVAN WITHIN-USER RANK AUDIT')
display(frames['07o_pavan_rank'])

## 8. Final production-policy snapshot

The final policy is allowed to stay unchanged even when external heuristics select additional plausible candidates.

The reason is that the project distinguishes:

```text
candidate plausibility
from
candidate reliability / persistence
from
production emission policy
```

In [ ]:
final_policy=s07p.build_final_policy_snapshot(frames)

assert int(final_policy.iloc[0]['production_HOME'])==27
assert int(final_policy.iloc[0]['production_OFFICE'])==16
assert final_policy.iloc[0]['final_policy']=='freeze_HOME_27_OFFICE_16'
assert not bool(final_policy.iloc[0]['change_home_gate'])
assert not bool(final_policy.iloc[0]['change_office_gate'])
assert not bool(final_policy.iloc[0]['promote_near_miss'])

display(final_policy)

## 9. Claim boundary

The project has no direct GeoLife HOME/OFFICE ground truth.

The synthesis therefore ends with an explicit **claim ledger**: what the accumulated evidence supports, and what it does not support.

In [ ]:
claim_boundary=s07p.build_claim_boundary()
display(claim_boundary)

## 10. Final methodological conclusion

The evidence hierarchy supports the following project-level interpretation:

### HOME

HOME is the more stable semantic target.

- internal fixed-window candidates converge strongly with alternate HOME rankers;
- Trackintel OSNA matches the frozen HOME-27 candidate identity exactly when the upstream representation is held fixed;
- independent literature-style HOME heuristics also converge strongly;
- end-to-end Trackintel differences are explainable largely through a smaller upstream stay inventory.

Therefore HOME-27 is a **conservative, evidence-supported emission set**, not the full set of plausible home candidates.

### OFFICE

OFFICE is structurally harder.

- schedule-aware rankers can agree strongly, but recurrence-style alternate-location ranking agrees weakly;
- held-out and split stability are weaker than HOME;
- Trackintel and literature-style methods frequently select additional WORK-plausible candidates;
- one-step threshold relaxations expose plausible near-misses, especially share-near cases;
- however the near-miss cohort does not jointly converge on recurrence, split-half, held-out persistence, BCL work context, and historical imagery.

Therefore OFFICE-16 should be interpreted as an **abstention-heavy conservative policy**.

### Final policy

```text
HOME   = 27
OFFICE = 16

do not relax OFFICE margin globally
do not relax OFFICE share globally
do not promote the nine audited near-miss candidates
do not reinterpret comparator agreement as semantic accuracy
```

The validation track is complete when these conclusions reproduce from aggregate caches.

## 11. Save synthesis artifacts

Stage 07p writes aggregate-only outputs.

In [ ]:
home_evidence.to_csv(C/'home_evidence_ladder.csv',index=False)
office_evidence.to_csv(C/'office_evidence_ladder.csv',index=False)
near_miss_policy.to_csv(C/'near_miss_policy_summary.csv',index=False)
pipeline_sensitivity.to_csv(C/'pipeline_sensitivity_summary.csv',index=False)
final_policy.to_csv(C/'final_policy_snapshot.csv',index=False)
claim_boundary.to_csv(C/'claim_boundary.csv',index=False)
lineage_validation.to_csv(C/'lineage_validation.csv',index=False)

print('saved:',C)